# method-override (Express Middleware)

**Tags:** #expressjs #nodejs #middleware #rest

`method-override` lets you use `PUT`, `PATCH`, and `DELETE` from places that only natively support `GET` and `POST` — chiefly HTML forms. It reads a query parameter, header, or body field, rewrites `req.method`, and lets Express route the request as if the real verb had been used.

---

## Why this exists

The HTML spec only allows `method="GET"` or `method="POST"` on a `<form>`. This isn't an Express limitation or a browser bug — it's the form spec itself, and it has never changed.

So if you're building a server-rendered CRUD app with EJS templates, you have a mismatch:

- REST says "delete this resource" = `DELETE /resource/1`
- HTML forms say "you may only send GET or POST"

`method-override` bridges the gap. The browser sends a real POST; the middleware rewrites it to DELETE before your router sees it, so you can write `app.delete(...)` and keep your routes RESTful.

> [!note] This is a server-rendered-app tool
> If your frontend talks to the server with `fetch()` or Axios, you don't need this — those APIs can send any verb natively. `method-override` is for the form-submission world (and the occasional corporate proxy that strips non-standard verbs).

---

## Setup

```bash
npm install method-override
```

```javascript
const express = require('express');
const methodOverride = require('method-override');
const app = express();

// Parse form bodies
app.use(express.urlencoded({ extended: true }));

// Look for ?_method=DELETE in the query string
app.use(methodOverride('_method'));
```

ESM:

```javascript
import methodOverride from 'method-override';
```

---

## Using it in a form

Set the form's real method to `POST` and put the intended verb in the **query string of the action URL**:

```html
<form action="/resource/1?_method=DELETE" method="POST">
  <button type="submit">Delete Item</button>
</form>
```

```html
<form action="/resource/1?_method=PUT" method="POST">
  <input name="title" />
  <button type="submit">Update</button>
</form>
```

Express then routes to:

```javascript
app.delete('/resource/:id', (req, res) => { /* ... */ });
app.put('/resource/:id', (req, res) => { /* ... */ });
```

> [!warning] The `_method` must be in the query string, not a hidden input
> With `methodOverride('_method')` the value is read from `req.query`. A `<input type="hidden" name="_method" value="DELETE">` lands in `req.body` and will be **silently ignored** — the request stays a POST and you get a 404 with no obvious cause. If you want the hidden-input style, you need the custom-function form below.

---

## The three sources

### 1. Query string (most common for forms)

```javascript
app.use(methodOverride('_method'));
```

Any getter string that doesn't start with `X-` is treated as a query key.

### 2. Header (for AJAX clients and restrictive proxies)

```javascript
app.use(methodOverride('X-HTTP-Method-Override'));
```

A getter starting with `X-` is treated as a header name. This is also the package default if you pass no argument at all.

### 3. Request body (needs a custom function)

```javascript
app.use(express.urlencoded({ extended: true })); // MUST come first

app.use(methodOverride((req, res) => {
  if (req.body && typeof req.body === 'object' && '_method' in req.body) {
    const method = req.body._method;
    delete req.body._method;   // strip it so it doesn't reach your handler
    return method;
  }
}));
```

This is what enables the hidden-input pattern. Note the ordering requirement — the body parser has to run first, or `req.body` is undefined and the override never fires.

---

## Middleware ordering

This bites people constantly. `app.use(methodOverride(...))` must appear **before your route definitions**. Express runs middleware in registration order; if your routes are mounted first, they see the unmodified POST.

```javascript
app.use(express.urlencoded({ extended: true }));
app.use(methodOverride('_method'));

// routes come AFTER
app.get('/resource', ...);
app.delete('/resource/:id', ...);
```

Full ordering rule: **body parser → method-override → routes.**

---

## Only POST gets overridden (by default)

The middleware has a `methods` option that defaults to `['POST']` — meaning only an original POST request is eligible to be rewritten.

```javascript
app.use(methodOverride('_method', { methods: ['POST'] })); // the default
```

This default exists for a reason. If GET requests could be overridden, then `<img src="/account/1?_method=DELETE">` on any page on the internet would delete your user's account. Browsers fire GETs for images, prefetch, and link previews without user intent.

**Don't widen this to include GET.** If you find yourself wanting to, the actual fix is almost always to use a form or a fetch call instead.

---

## Security notes

**CSRF still applies.** `method-override` does nothing to authenticate the request — it only rewrites the verb. A cross-site form can still POST to your endpoint and have it turn into a DELETE. Use `csurf` (or, more practically now, `SameSite=Lax` cookies plus a token) exactly as you would for any state-changing POST.

**Strip the override field.** In the body-based version above, `delete req.body._method` matters — otherwise `_method` leaks into your handler and can end up in a `Model.create(req.body)` call or a database column.

**Don't trust the verb for authorization.** Authorize on the route handler and the user's identity, not on "this arrived as a DELETE so it must be legitimate."

---

## Common failure modes

| Symptom | Cause |
|---|---|
| Route returns 404, request logs as POST | Middleware registered after routes |
| Hidden `_method` input ignored | Using `methodOverride('_method')`, which reads the query string only |
| `req.body` is undefined in custom getter | Body parser not registered before method-override |
| `_method` appears in your database | Forgot to `delete req.body._method` |
| Works in dev, breaks behind a proxy | Proxy stripping the `X-HTTP-Method-Override` header |

---

## Debugging it

Drop this above your routes to see what's actually happening:

```javascript
app.use((req, res, next) => {
  console.log(req.method, req.originalUrl, req.query, req.body);
  next();
});
```

Place it once **before** `methodOverride` and once **after** — if the method is still POST in the second log, the override isn't firing and the problem is ordering or the wrong source.

---

## References

- [method-override on npm](https://www.npmjs.com/package/method-override)
- [Express middleware docs](https://expressjs.com/en/resources/middleware/method-override/)